# 08 — E2E Cross-Validation v2 (plan_2)
**SentinelFatal2 · Google Colab notebook**

SSOT: `docs/plan_2.md` | Workflow: `docs/planWorkflow_2.md`

Pipeline:
1. Shared pretrain on 687 recordings
2. Config selection A/B/C on 441/56 split
3. Fold 0 pilot
4. G0 ablation (shared vs clean pretrain, folds 0+1)
5. Folds 1..4 (resume-aware)
6. AT sweep summary + feature gate
7. Bootstrap CI + comparison table
8. Reproducibility checks + Reproducibility Track (441/56/55)
9. Export to Drive / GitHub

**Rules:** No official metrics with stride=1. No threshold tuning on test fold. Guard clauses on all data counts.

In [1]:
# ── Cell 1: Runtime / Seed / Determinism (plan_2 §15.1) ──────────────────────
import sys, os, random, time
import numpy as np

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

import torch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Determinism — mandatory per plan_2 §15.1
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print(f'[SEED]  random={SEED}  numpy={SEED}  torch={SEED}  cuda_all={SEED}')
print(f'[DETER] cudnn.deterministic={torch.backends.cudnn.deterministic}')
print(f'[DETER] cudnn.benchmark={torch.backends.cudnn.benchmark}')
print(f'[GPU]   available={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'[GPU]   device={torch.cuda.get_device_name(0)}  '
          f'mem={torch.cuda.get_device_properties(0).total_memory // (1<<30)} GB')
print(f'[PY]    {sys.version.split()[0]}  torch={torch.__version__}')
sys.stdout.flush()

[SEED]  random=42  numpy=42  torch=42  cuda_all=42
[DETER] cudnn.deterministic=True
[DETER] cudnn.benchmark=False
[GPU]   available=True
[GPU]   device=Tesla T4  mem=14 GB
[PY]    3.12.12  torch=2.10.0+cu128


In [2]:
# ── Cell 2: Clone / Pull repo ─────────────────────────────────────────────────
import subprocess, sys, os, pathlib

REPO_URL = 'https://github.com/ArielShamay/SentinelFatal2.git'
BRANCH   = 'master'

IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
ROOT     = '/content/SentinelFatal2' if IS_COLAB else str(
    pathlib.Path('__file__' in dir() and __file__ or '.').resolve().parent.parent)

def sh(cmd, check=True):
    print(f'$ {cmd}'); sys.stdout.flush()
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip(), file=sys.stderr)
    if check: r.check_returncode()
    return r

if IS_COLAB:
    if not os.path.isdir(ROOT):
        sh(f'git clone --branch {BRANCH} {REPO_URL} {ROOT}')
    else:
        sh(f'git -C {ROOT} pull origin {BRANCH}')
    sh(f'pip install -q -r {ROOT}/requirements.txt')

sys.path.insert(0, ROOT)
print(f'[OK]  ROOT={ROOT}  is_colab={IS_COLAB}')
sys.stdout.flush()

$ git clone --branch master https://github.com/ArielShamay/SentinelFatal2.git /content/SentinelFatal2


KeyboardInterrupt: 

In [ ]:
# ── Cell 3: GPU preflight ──────────────────────────────────────────────────────
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if DEVICE == 'cuda':
    sh('nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader')
    x = torch.randn(4, 2, 1800, device=DEVICE)
    assert x.shape == (4, 2, 1800), 'GPU smoke test failed'
    del x; torch.cuda.empty_cache()
    print('[OK]  GPU smoke test passed')
else:
    print('[WARN] No CUDA GPU — training will be very slow on CPU')

print(f'[OK]  torch={torch.__version__}  CUDA={torch.version.cuda}  DEVICE={DEVICE}')
sys.stdout.flush()

In [ ]:
# ── Cell 4: Download data_processed.zip from GitHub ──────────────────────────
# Primary: GitHub raw URL (A1 verified PASS, SHA-256 documented in data_pack_check.md)
import pathlib

ROOT_P         = pathlib.Path(ROOT)
DATA_ZIP_PATH  = ROOT_P / 'data_processed.zip'
PROC_ROOT      = ROOT_P / 'data' / 'processed'
GITHUB_ZIP_URL = 'https://raw.githubusercontent.com/ArielShamay/SentinelFatal2/master/data_processed.zip'

ctu_dir    = PROC_ROOT / 'ctu_uhb'
fhrma_dir  = PROC_ROOT / 'fhrma'
n_ctu_now  = len([f for f in ctu_dir.glob('*.npy')   if f.name != '.gitkeep']) if ctu_dir.exists()   else 0
n_fhrma_now = len([f for f in fhrma_dir.glob('*.npy') if f.name != '.gitkeep']) if fhrma_dir.exists() else 0

if n_ctu_now >= 552 and n_fhrma_now >= 135:
    print(f'[OK]  Processed data already present — skipping download')
    print(f'      CTU-UHB={n_ctu_now}  FHRMA={n_fhrma_now}')
else:
    print(f'[download] Downloading from GitHub: {GITHUB_ZIP_URL}')
    sys.stdout.flush()
    data_dir = ROOT_P / 'data'
    sh(f'wget -q -O "{DATA_ZIP_PATH}" "{GITHUB_ZIP_URL}"')
    sh(f'unzip -q "{DATA_ZIP_PATH}" -d "{data_dir}"')
    if DATA_ZIP_PATH.exists():
        DATA_ZIP_PATH.unlink()
    print(f'[OK]  Extracted to {PROC_ROOT}')

sys.stdout.flush()

In [ ]:
# ── Cell 5: Count sanity (CTU-UHB=552, FHRMA=135) ────────────────────────────
import pathlib

ROOT_P    = pathlib.Path(ROOT)
CTU_DIR   = ROOT_P / 'data' / 'processed' / 'ctu_uhb'
FHRMA_DIR = ROOT_P / 'data' / 'processed' / 'fhrma'

n_ctu   = len([f for f in CTU_DIR.glob('*.npy')   if f.name != '.gitkeep'])
n_fhrma = len([f for f in FHRMA_DIR.glob('*.npy') if f.name != '.gitkeep'])

print(f'[SanityCheck]  CTU-UHB: {n_ctu}/552   FHRMA: {n_fhrma}/135')

assert n_ctu   == 552, f'[FAIL] Expected 552 CTU-UHB npy files, got {n_ctu}'
assert n_fhrma == 135, f'[FAIL] Expected 135 FHRMA npy files, got {n_fhrma}'

print('[PASS] Count sanity OK')
sys.stdout.flush()

In [ ]:
# ── Cell 6: Shared pretrain (or load existing checkpoint) ────────────────────
import pathlib, pandas as pd
from src.train.pretrain import pretrain

ROOT_P               = pathlib.Path(ROOT)
config_path          = ROOT_P / 'config' / 'train_config.yaml'
shared_ckpt_dir      = ROOT_P / 'checkpoints' / 'e2e_cv_v2' / 'shared_pretrain'
shared_pretrain_ckpt = shared_ckpt_dir / 'best_pretrain.pt'
pretrain_log         = ROOT_P / 'logs' / 'e2e_cv_v2' / 'shared_pretrain_loss.csv'

shared_ckpt_dir.mkdir(parents=True, exist_ok=True)
(ROOT_P / 'logs' / 'e2e_cv_v2').mkdir(parents=True, exist_ok=True)

if shared_pretrain_ckpt.exists():
    print(f'[pretrain] Checkpoint exists — skipping: {shared_pretrain_ckpt}')
else:
    print('[pretrain] Starting shared pretrain on all 687 recordings...')
    sys.stdout.flush()
    pretrain(
        config_path    = config_path,
        device_str     = DEVICE,
        max_batches    = 0,
        pretrain_csv   = str(ROOT_P / 'data' / 'splits' / 'pretrain.csv'),
        processed_root = str(ROOT_P / 'data' / 'processed'),
        checkpoint_dir = str(shared_ckpt_dir),
        log_path       = str(pretrain_log),
    )

print(f'[pretrain] DONE — checkpoint: {shared_pretrain_ckpt}')

# G1 gate
if pretrain_log.exists():
    pt_df    = pd.read_csv(pretrain_log)
    best_mse = float(pt_df['val_loss'].min()) if 'val_loss' in pt_df.columns else 1.0
    ok = best_mse < 0.015
    print(f'[GATE G1a] {"PASS" if ok else "FAIL"} — best_val_mse={best_mse:.5f} < 0.015')
    if not ok:
        print('[G1] FAIL: val_mse >= 0.015 — consider more epochs or different scheduler')

sys.stdout.flush()

In [ ]:
# ── Cell 7: Config selection A/B/C on 441/56 (plan_2 §3.3) ───────────────────
# Fallback to Config A if time budget < 90 min (plan_2 §7.1).
# RULE: config chosen on 441/56 ONLY — never on test fold.
import json, time as _time, torch, pathlib, pandas as pd
from src.train.finetune  import train as finetune_train, load_pretrained_checkpoint
from src.train.utils     import compute_recording_auc
from src.model.patchtst  import PatchTST, load_config
from src.model.heads     import ClassificationHead

ROOT_P             = pathlib.Path(ROOT)
config_path        = ROOT_P / 'config' / 'train_config.yaml'
results_base       = ROOT_P / 'results' / 'e2e_cv_v2'
locked_config_path = results_base / 'locked_config.json'
log_cfg_path       = ROOT_P / 'logs' / 'e2e_cv_v2' / 'config_selection_441_56.csv'
results_base.mkdir(parents=True, exist_ok=True)

CONFIG_CANDIDATES = {
    'A': {'loss': 'cross_entropy'},
    'B': {'loss': 'focal', 'focal_gamma': 2.0},
    'C': {'d_model': 256, 'n_layers': 4, 'loss': 'focal', 'focal_gamma': 2.0},
}
TIME_LIMIT_SEC = 90 * 60

if locked_config_path.exists():
    with open(locked_config_path) as f:
        locked_config = json.load(f)
    print(f'[config] Locked config loaded: {locked_config}')
else:
    train_csv_sel = ROOT_P / 'data' / 'splits' / 'train.csv'
    val_csv_sel   = ROOT_P / 'data' / 'splits' / 'val.csv'
    best_name, best_auc = 'A', 0.0
    cfg_rows, t_start   = [], _time.time()

    for cfg_name, overrides in CONFIG_CANDIDATES.items():
        elapsed = _time.time() - t_start
        if elapsed > TIME_LIMIT_SEC:
            print(f'[config] TIME LIMIT {elapsed/60:.1f} min — falling back to Config A')
            best_name = 'A'; break

        print(f'\n[config] Testing {cfg_name}: {overrides}')
        sys.stdout.flush()
        cfg_dir = ROOT_P / 'checkpoints' / 'e2e_cv_v2' / f'config_{cfg_name}'
        cfg_dir.mkdir(parents=True, exist_ok=True)

        finetune_train(
            config_path=config_path, device_str=DEVICE, max_batches=0,
            processed_root=str(ROOT_P / 'data' / 'processed'),
            train_csv=str(train_csv_sel), val_csv=str(val_csv_sel),
            pretrain_checkpoint=str(shared_pretrain_ckpt),
            checkpoint_dir=str(cfg_dir),
            log_path=str(cfg_dir / 'finetune_loss.csv'),
            config_overrides=overrides,
        )

        cfg_m = load_config(config_path, overrides=overrides)
        model = PatchTST(cfg_m).to(DEVICE)
        d_in  = (int(cfg_m['data']['n_patches']) *
                 int(cfg_m['model']['d_model'])  *
                 int(cfg_m['data']['n_channels']))
        model.replace_head(ClassificationHead(
            d_in=d_in, n_classes=int(cfg_m['finetune']['n_classes']),
            dropout=float(cfg_m['model']['dropout'])))
        load_pretrained_checkpoint(model, cfg_dir / 'best_finetune.pt', torch.device(DEVICE))

        val_auc = compute_recording_auc(
            model, val_csv_sel, ROOT_P / 'data' / 'processed', stride=60, device=DEVICE)
        print(f'[config {cfg_name}] val_auc={val_auc:.4f}')
        cfg_rows.append({'config': cfg_name, 'val_auc': round(val_auc, 4)})
        sys.stdout.flush()
        if val_auc > best_auc:
            best_auc, best_name = val_auc, cfg_name

    ok = best_auc >= 0.70
    print(f'\n[GATE G2] {"PASS" if ok else "FAIL"} — best_val_auc={best_auc:.4f} >= 0.70')
    if not ok:
        print('[G2] FAIL: forcing fallback to Config A')
        best_name = 'A'

    locked_config = {'name': best_name, 'val_auc': round(best_auc, 4)}
    with open(locked_config_path, 'w') as f:
        json.dump(locked_config, f, indent=2)
    pd.DataFrame(cfg_rows).to_csv(log_cfg_path, index=False)
    print(f'[config] Locked to {best_name} (val_auc={best_auc:.4f}) -> {locked_config_path}')

sys.stdout.flush()

In [ ]:
# ── Cell 8: Fold 0 pilot end-to-end ───────────────────────────────────────────
import pathlib, pandas as pd
from sklearn.metrics         import roc_auc_score
from scripts.run_e2e_cv_v2  import generate_cv_splits, run_fold

ROOT_P         = pathlib.Path(ROOT)
config_path    = ROOT_P / 'config' / 'train_config.yaml'
results_base   = ROOT_P / 'results'     / 'e2e_cv_v2'
ckpt_base      = ROOT_P / 'checkpoints' / 'e2e_cv_v2'
processed_root = ROOT_P / 'data'        / 'processed'
splits_dir     = ROOT_P / 'data'        / 'splits'
results_base.mkdir(parents=True, exist_ok=True)

# Build all-552 CSV if not present (merge train/val/test)
all_csv = splits_dir / 'train_val_test.csv'
if not all_csv.exists():
    dfs = [pd.read_csv(splits_dir / fn, dtype={'id': str, 'target': int})
           for fn in ['train.csv', 'val.csv', 'test.csv']
           if (splits_dir / fn).exists()]
    assert dfs, f'No split CSVs in {splits_dir}'
    all_df = pd.concat(dfs, ignore_index=True).drop_duplicates('id')
    assert len(all_df) == 552, f'Expected 552, got {len(all_df)}'
    all_df.to_csv(all_csv, index=False)
    print(f'[splits] Created {all_csv}: {len(all_df)} recordings')

fold_splits = generate_cv_splits(all_csv, n_folds=5, seed=42)
print(f'[splits] 5 folds over {len(fold_splits[0]["df_all"])} recordings')

# Fold 0 pilot
oof_fold0 = results_base / 'fold0_oof_scores.csv'
if oof_fold0.exists():
    print('[fold 0] OOF CSV found — loading from cache')
    oof0 = pd.read_csv(oof_fold0)
    fold0_auc = roc_auc_score(oof0['y_true'], oof0['y_score']) if oof0['y_true'].nunique() > 1 else 0.
    print(f'[fold 0] Cached AUC={fold0_auc:.4f}')
else:
    print('[fold 0] Starting pilot run...')
    sys.stdout.flush()
    result0 = run_fold(
        fold_k=0, split=fold_splits[0],
        processed_root=processed_root, config_path=config_path,
        shared_pretrain_ckpt=ckpt_base / 'shared_pretrain' / 'best_pretrain.pt',
        checkpoint_base=ckpt_base, results_base=results_base,
        device=DEVICE, dry_run=False,
    )
    fold0_auc = result0['lr_test_auc']
    print(f'[fold 0] DONE — lr_test_auc={fold0_auc:.4f}')

ok = fold0_auc >= 0.65
print(f'[GATE G3] {"PASS" if ok else "FAIL"} — fold0 AUC={fold0_auc:.4f} >= 0.65')
if not ok:
    print('[G3] FAIL — diagnose before running folds 1-4')
sys.stdout.flush()

In [ ]:
# ── Cell 9: G0 ablation — shared vs clean pretrain (folds 0+1) ───────────────
# Mandatory before continuing to folds 2-4.
import pathlib, pandas as pd
from scripts.run_e2e_cv_v2 import run_g0_ablation

ROOT_P       = pathlib.Path(ROOT)
results_base = ROOT_P / 'results'     / 'e2e_cv_v2'
ckpt_base    = ROOT_P / 'checkpoints' / 'e2e_cv_v2'
g0_csv       = results_base / 'ablation_shared_vs_clean.csv'

if g0_csv.exists():
    print('[G0] ablation_shared_vs_clean.csv found — loading from cache')
    g0_df      = pd.read_csv(g0_csv)
    mean_delta = float(g0_df['delta_auc'].abs().mean())
    g0_passed  = mean_delta <= 0.01
    print(g0_df.to_string(index=False))
else:
    print('[G0] Running ablation (shared vs clean pretrain on folds 0+1)...')
    sys.stdout.flush()
    mean_delta, g0_passed = run_g0_ablation(
        fold_splits=fold_splits,
        processed_root=ROOT_P / 'data' / 'processed',
        config_path=ROOT_P / 'config' / 'train_config.yaml',
        device=DEVICE,
        shared_pretrain_ckpt=ckpt_base / 'shared_pretrain' / 'best_pretrain.pt',
        checkpoint_base=ckpt_base,
        out_path=g0_csv,
        dry_run=False,
    )

print(f'[GATE G0] {"PASS" if g0_passed else "FAIL/WARN"} — mean|dAUC|={mean_delta:.4f} <= 0.01')
if not g0_passed:
    print('[G0] WARN: transductive leak detected. Continue with mandatory disclosure.')
sys.stdout.flush()

In [ ]:
# ── Cell 10: Folds 1..4 loop (resume-aware) ──────────────────────────────────
import pathlib, numpy as np, pandas as pd
from sklearn.metrics        import roc_auc_score
from scripts.run_e2e_cv_v2 import run_fold

ROOT_P         = pathlib.Path(ROOT)
results_base   = ROOT_P / 'results'     / 'e2e_cv_v2'
ckpt_base      = ROOT_P / 'checkpoints' / 'e2e_cv_v2'
processed_root = ROOT_P / 'data'        / 'processed'
config_path    = ROOT_P / 'config'      / 'train_config.yaml'

fold_results, oof_dfs = [], []

# Collect fold 0 from Cell 8
f0_oof = results_base / 'fold0_oof_scores.csv'
if f0_oof.exists():
    oof0 = pd.read_csv(f0_oof)
    auc0 = roc_auc_score(oof0['y_true'], oof0['y_score']) if oof0['y_true'].nunique() > 1 else 0.
    fold_results.append({'fold': 0, 'lr_test_auc': round(auc0, 4),
                         'n_test': len(oof0), 'n_pos_test': int(oof0['y_true'].sum())})
    oof_dfs.append(oof0)

# Folds 1..4
for fold_k in range(1, 5):
    oof_path = results_base / f'fold{fold_k}_oof_scores.csv'
    if oof_path.exists():
        print(f'[fold {fold_k}] RESUME — loading cached OOF')
        oof_df = pd.read_csv(oof_path)
        auc_k  = roc_auc_score(oof_df['y_true'], oof_df['y_score']) if oof_df['y_true'].nunique() > 1 else 0.
        fold_results.append({'fold': fold_k, 'lr_test_auc': round(auc_k, 4),
                             'n_test': len(oof_df), 'n_pos_test': int(oof_df['y_true'].sum())})
        oof_dfs.append(oof_df)
        print(f'[fold {fold_k}] Cached AUC={auc_k:.4f}')
        continue

    print(f'\n[fold {fold_k}] Starting...')
    sys.stdout.flush()
    try:
        result = run_fold(
            fold_k=fold_k, split=fold_splits[fold_k],
            processed_root=processed_root, config_path=config_path,
            shared_pretrain_ckpt=ckpt_base / 'shared_pretrain' / 'best_pretrain.pt',
            checkpoint_base=ckpt_base, results_base=results_base,
            device=DEVICE, dry_run=False,
        )
        fold_results.append(result)
        oof_dfs.append(pd.read_csv(results_base / f'fold{fold_k}_oof_scores.csv'))
        print(f'[fold {fold_k}] DONE — lr_test_auc={result["lr_test_auc"]:.4f}')
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f'[fold {fold_k}] ERROR: {e} — see incidents.md')

# G4 gate
aucs     = [r['lr_test_auc'] for r in fold_results if 'lr_test_auc' in r]
mean_auc = float(np.mean(aucs)) if aucs else 0.
std_auc  = float(np.std(aucs))  if aucs else 1.
print(f'\n[GATE G4a] {"PASS" if mean_auc >= 0.70 else "FAIL"} — mean_auc={mean_auc:.4f} >= 0.70')
print(f'[GATE G4b] {"PASS" if std_auc  < 0.10 else "FAIL"} — std_auc ={std_auc:.4f} < 0.10')
sys.stdout.flush()

In [ ]:
# ── Cell 11: AT sweep summary -> threshold_sweep_summary.csv ─────────────────
# AT sweep runs inside run_fold; this cell collects and saves the summary.
# AT_candidates = [0.30, 0.35, 0.40, 0.45] (plan_2 §4.2)
import pathlib, pandas as pd

ROOT_P       = pathlib.Path(ROOT)
results_base = ROOT_P / 'results' / 'e2e_cv_v2'
out_path     = results_base / 'threshold_sweep_summary.csv'

rows = []
for fold_k in range(5):
    oof_path = results_base / f'fold{fold_k}_oof_scores.csv'
    if not oof_path.exists():
        continue
    oof = pd.read_csv(oof_path)
    best_at = float(oof['best_at'].iloc[0])           if 'best_at'           in oof.columns else 0.40
    th_pri  = float(oof['threshold_primary'].iloc[0]) if 'threshold_primary' in oof.columns else 0.5
    rows.append({
        'fold': fold_k, 'best_at': best_at,
        'threshold_primary': round(th_pri, 4),
        'n_test': len(oof), 'n_pos_test': int(oof['y_true'].sum()),
    })

if rows:
    sweep_df = pd.DataFrame(rows)
    sweep_df.to_csv(out_path, index=False)
    print(f'[AT sweep] threshold_sweep_summary.csv -> {out_path}')
    print(sweep_df.to_string(index=False))
else:
    print('[AT sweep] No OOF files found yet — run Cells 8-10 first')
sys.stdout.flush()

In [ ]:
# ── Cell 12: Feature gate — 6 vs 12 features (plan_2 §4.1) ──────────────────
# MVP baseline: LR + 6 features (Phase A default).
# Gate on ft_val of folds 0+1: PASS if mean_delta >= 0.01 AND no single-fold degradation.
# Output: features_6_vs_12_gate.csv
import pathlib, numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score
from src.model.patchtst  import PatchTST, load_config
from src.model.heads     import ClassificationHead
from src.train.finetune  import load_pretrained_checkpoint
from scripts.run_e2e_cv_v2 import extract_features_for_split, fit_lr_model, predict_lr

ROOT_P         = pathlib.Path(ROOT)
config_path    = ROOT_P / 'config' / 'train_config.yaml'
results_base   = ROOT_P / 'results'     / 'e2e_cv_v2'
ckpt_base      = ROOT_P / 'checkpoints' / 'e2e_cv_v2'
processed_root = ROOT_P / 'data'        / 'processed'
gate_csv       = results_base / 'features_6_vs_12_gate.csv'

rows = []
for fold_k in [0, 1]:
    ft_ckpt   = ckpt_base  / f'fold{fold_k}' / 'finetune' / 'best_finetune.pt'
    sp_dir    = results_base / f'fold{fold_k}_splits'
    train_csv = sp_dir / 'train.csv'
    val_csv   = sp_dir / 'val.csv'

    if not (ft_ckpt.exists() and train_csv.exists() and val_csv.exists()):
        print(f'[feat_gate] fold {fold_k}: missing artifacts — skipping')
        continue

    cfg_m = load_config(config_path)
    model = PatchTST(cfg_m).to(DEVICE)
    d_in  = (int(cfg_m['data']['n_patches']) *
             int(cfg_m['model']['d_model'])  *
             int(cfg_m['data']['n_channels']))
    model.replace_head(ClassificationHead(
        d_in=d_in, n_classes=int(cfg_m['finetune']['n_classes']),
        dropout=float(cfg_m['model']['dropout'])))
    load_pretrained_checkpoint(model, ft_ckpt, torch.device(DEVICE))
    model.eval()

    for n_feat in [6, 12]:
        X_tr, y_tr, _ = extract_features_for_split(model, train_csv, processed_root, 0.40, 24, DEVICE, n_feat)
        X_vl, y_vl, _ = extract_features_for_split(model, val_csv,   processed_root, 0.40, 24, DEVICE, n_feat)
        if len(np.unique(y_tr)) < 2 or len(np.unique(y_vl)) < 2:
            rows.append({'fold': fold_k, 'n_features': n_feat, 'ft_val_auc': 0.0})
            continue
        sc, pca, lr_m = fit_lr_model(X_tr, y_tr, C=0.1, use_pca=(n_feat == 12))
        val_auc = roc_auc_score(y_vl, predict_lr(X_vl, sc, pca, lr_m))
        rows.append({'fold': fold_k, 'n_features': n_feat, 'ft_val_auc': round(val_auc, 4)})
        print(f'  fold{fold_k}  n_features={n_feat}  ft_val_auc={val_auc:.4f}')

gate_df = pd.DataFrame(rows)
if not gate_df.empty:
    pivot = gate_df.pivot(index='fold', columns='n_features', values='ft_val_auc').reset_index()
    pivot.columns.name = None
    pivot = pivot.rename(columns={6: 'auc_6feat', 12: 'auc_12feat'})
    pivot['delta_auc']     = (pivot['auc_12feat'] - pivot['auc_6feat']).round(4)
    mean_delta             = pivot['delta_auc'].mean()
    no_degradation         = (pivot['delta_auc'] >= 0).all()
    gate_pass_12           = (mean_delta >= 0.01) and no_degradation
    pivot['gate_decision'] = 'PASS_12feat' if gate_pass_12 else 'ROLLBACK_6feat'
    pivot.to_csv(gate_csv, index=False)
    print(f'\n[feat_gate] mean_delta={mean_delta:.4f}  no_degradation={no_degradation}')
    decision = 'PASS - use 12 features' if gate_pass_12 else 'FAIL - rollback to 6 features (MVP default)'
    print(f'[feat_gate] Decision: {decision}')
    print(pivot.to_string(index=False))
else:
    print('[feat_gate] No data — run Cells 8-10 first')
sys.stdout.flush()

In [ ]:
# ── Cell 13: Bootstrap CI + final tables + comparison_table.csv ──────────────
import pathlib, numpy as np, pandas as pd
from sklearn.metrics        import roc_auc_score
from scripts.run_e2e_cv_v2 import bootstrap_auc_ci, N_BOOTSTRAP, SEED

ROOT_P       = pathlib.Path(ROOT)
results_base = ROOT_P / 'results' / 'e2e_cv_v2'

oof_dfs_all = []
for fold_k in range(5):
    p = results_base / f'fold{fold_k}_oof_scores.csv'
    if p.exists():
        oof_dfs_all.append(pd.read_csv(p))

if not oof_dfs_all:
    print('[WARN] No OOF data — run Cells 8-10 first')
else:
    oof_all = pd.concat(oof_dfs_all, ignore_index=True)
    oof_all.to_csv(results_base / 'global_oof_predictions.csv', index=False)

    global_auc = roc_auc_score(oof_all['y_true'], oof_all['y_score']) if oof_all['y_true'].nunique() > 1 else 0.
    ci_lo, ci_hi = bootstrap_auc_ci(
        oof_all['y_true'].values, oof_all['y_score'].values,
        n_bootstrap=N_BOOTSTRAP, seed=SEED)

    print(f'=====================================================')
    print(f'  Global OOF AUC : {global_auc:.4f}')
    print(f'  95% CI         : [{ci_lo:.4f}, {ci_hi:.4f}]  (width={ci_hi-ci_lo:.4f})')
    print(f'  N recordings   : {len(oof_all)}')
    print(f'  N acidemia     : {int(oof_all["y_true"].sum())}')
    print(f'=====================================================')

    # Per-fold table
    per_fold_rows = []
    for i, oof_k in enumerate(oof_dfs_all):
        auc_k = roc_auc_score(oof_k['y_true'], oof_k['y_score']) if oof_k['y_true'].nunique() > 1 else 0.
        per_fold_rows.append({'fold': i, 'lr_test_auc': round(auc_k, 4),
                              'n_test': len(oof_k), 'n_pos_test': int(oof_k['y_true'].sum())})
    per_fold_df = pd.DataFrame(per_fold_rows)
    per_fold_df.to_csv(results_base / 'e2e_cv_v2_per_fold.csv', index=False)

    pd.DataFrame([{
        'global_oof_auc': round(global_auc, 4), 'ci_95_lo': round(ci_lo, 4),
        'ci_95_hi': round(ci_hi, 4), 'ci_width': round(ci_hi - ci_lo, 4),
        'n_folds': 5, 'n_recordings': len(oof_all),
        'n_pos': int(oof_all['y_true'].sum()), 'n_bootstrap': N_BOOTSTRAP,
    }]).to_csv(results_base / 'e2e_cv_v2_final_report.csv', index=False)

    # Comparison table (plan_2 §9.3)
    repro_auc, repro_lo, repro_hi = None, None, None
    repro_csv = results_base / 'repro_track_55.csv'
    if repro_csv.exists():
        repro_df = pd.read_csv(repro_csv)
        if 'y_true' in repro_df.columns and 'y_score' in repro_df.columns:
            repro_auc = round(roc_auc_score(repro_df['y_true'], repro_df['y_score']), 4)
            r_lo, r_hi = bootstrap_auc_ci(
                repro_df['y_true'].values, repro_df['y_score'].values,
                n_bootstrap=5000, seed=SEED)
            repro_lo, repro_hi = round(r_lo, 4), round(r_hi, 4)

    comparison = pd.DataFrame([
        {'method': 'Paper (benchmark)',  'n': 55,  'auc': 0.826, 'ci_lo': None,     'ci_hi': None},
        {'method': 'Baseline (test-55)', 'n': 55,  'auc': 0.812, 'ci_lo': 0.630,    'ci_hi': 0.953},
        {'method': 'Repro Track',        'n': 55,  'auc': repro_auc, 'ci_lo': repro_lo, 'ci_hi': repro_hi},
        {'method': 'E2E CV (552)',       'n': 552, 'auc': round(global_auc, 4),
         'ci_lo': round(ci_lo, 4), 'ci_hi': round(ci_hi, 4)},
    ])
    comparison.to_csv(results_base / 'comparison_table.csv', index=False)
    print('\n[comparison_table]')
    print(comparison.to_string(index=False))

sys.stdout.flush()

In [ ]:
# ── Cell 14: Reproducibility checks (plan_2 §15.1) ────────────────────────────
# 1. SHA-256 of split CSVs -> split_hashes.txt
# 2. Add git_commit column to all result CSVs
# 3. Generate requirements.txt with exact package versions
import hashlib, subprocess, importlib, pathlib, pandas as pd

ROOT_P       = pathlib.Path(ROOT)
logs_base    = ROOT_P / 'logs'    / 'e2e_cv_v2'
results_base = ROOT_P / 'results' / 'e2e_cv_v2'

# 1. Split CSV hashes
split_hashes_path = logs_base / 'split_hashes.txt'
splits_dir        = ROOT_P / 'data' / 'splits'
with open(split_hashes_path, 'w') as fh:
    for fname in ['train.csv', 'val.csv', 'test.csv', 'pretrain.csv']:
        fpath = splits_dir / fname
        if fpath.exists():
            sha = hashlib.sha256(fpath.read_bytes()).hexdigest()
            fh.write(f'{sha}  {fname}\n')
            print(f'  {sha[:16]}...  {fname}')
        else:
            fh.write(f'MISSING  {fname}\n')
            print(f'  [WARN] Missing: {fname}')
print(f'[repro] split_hashes.txt -> {split_hashes_path}')

# 2. git_commit in all result CSVs
r = subprocess.run(f'git -C {ROOT} rev-parse HEAD',
                   shell=True, capture_output=True, text=True)
git_commit = r.stdout.strip()[:12] if r.returncode == 0 else 'unknown'
print(f'[repro] git_commit = {git_commit}')

for csv_path in results_base.glob('*.csv'):
    try:
        df = pd.read_csv(csv_path)
        if 'git_commit' not in df.columns:
            df['git_commit'] = git_commit
            df.to_csv(csv_path, index=False)
            print(f'  added git_commit -> {csv_path.name}')
    except Exception as e:
        print(f'  [WARN] {csv_path.name}: {e}')

# 3. requirements.txt with exact versions
pkg_map = {'numpy': 'numpy', 'torch': 'torch', 'sklearn': 'scikit-learn',
           'pandas': 'pandas', 'scipy': 'scipy', 'joblib': 'joblib'}
reqs = []
for mod_name, pkg_name in pkg_map.items():
    try:
        mod = importlib.import_module(mod_name)
        ver = getattr(mod, '__version__', '?')
        reqs.append(f'{pkg_name}=={ver}')
    except ImportError:
        reqs.append(f'# {pkg_name} not installed')

reqs_path = ROOT_P / 'requirements.txt'
reqs_path.write_text('\n'.join(reqs) + '\n')
print(f'[repro] requirements.txt -> {reqs_path}')
for r_line in reqs: print(f'  {r_line}')

sys.stdout.flush()

In [ ]:
# ── Cell 15: Reproducibility Track — 441/56/55 split (plan_2 §9.3) ───────────
# Runs full pipeline on the historical split with locked config.
# RULE: No test-time tuning. AT + threshold locked on ft_val=56 ONLY.
import pathlib, numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score
from src.train.finetune  import train as finetune_train, load_pretrained_checkpoint
from src.model.patchtst  import PatchTST, load_config
from src.model.heads     import ClassificationHead
from scripts.run_e2e_cv_v2 import (
    extract_features_for_split, fit_lr_model, predict_lr,
    at_sweep, clinical_threshold,
)

ROOT_P         = pathlib.Path(ROOT)
config_path    = ROOT_P / 'config' / 'train_config.yaml'
results_base   = ROOT_P / 'results'     / 'e2e_cv_v2'
ckpt_base      = ROOT_P / 'checkpoints' / 'e2e_cv_v2'
processed_root = ROOT_P / 'data'        / 'processed'
splits_dir     = ROOT_P / 'data'        / 'splits'
repro_csv      = results_base / 'repro_track_55.csv'

if repro_csv.exists():
    print(f'[repro_track] Already exists — loading: {repro_csv}')
    repro_df  = pd.read_csv(repro_csv)
    repro_auc = roc_auc_score(repro_df['y_true'], repro_df['y_score']) if repro_df['y_true'].nunique() > 1 else 0.
    print(f'[repro_track] AUC on test-55 = {repro_auc:.4f}')
else:
    train_csv = splits_dir / 'train.csv'
    val_csv   = splits_dir / 'val.csv'
    test_csv  = splits_dir / 'test.csv'
    for p in [train_csv, val_csv, test_csv]:
        assert p.exists(), f'[FAIL] Missing: {p}'

    repro_ft_dir  = ckpt_base / 'repro_track' / 'finetune'
    repro_ft_dir.mkdir(parents=True, exist_ok=True)
    repro_ft_ckpt = repro_ft_dir / 'best_finetune.pt'

    if not repro_ft_ckpt.exists():
        print('[repro_track] Finetuning on 441/56 split...')
        sys.stdout.flush()
        finetune_train(
            config_path=config_path, device_str=DEVICE, max_batches=0,
            processed_root=str(processed_root),
            train_csv=str(train_csv), val_csv=str(val_csv),
            pretrain_checkpoint=str(ckpt_base / 'shared_pretrain' / 'best_pretrain.pt'),
            checkpoint_dir=str(repro_ft_dir),
            log_path=str(repro_ft_dir / 'finetune_loss.csv'),
        )
    else:
        print(f'[repro_track] Finetune checkpoint exists: {repro_ft_ckpt}')

    cfg_m = load_config(config_path)
    model = PatchTST(cfg_m).to(DEVICE)
    d_in  = (int(cfg_m['data']['n_patches']) *
             int(cfg_m['model']['d_model'])  *
             int(cfg_m['data']['n_channels']))
    model.replace_head(ClassificationHead(
        d_in=d_in, n_classes=int(cfg_m['finetune']['n_classes']),
        dropout=float(cfg_m['model']['dropout'])))
    load_pretrained_checkpoint(model, repro_ft_ckpt, torch.device(DEVICE))
    model.eval()

    # AT sweep on ft_val=56 — Phase A uses 6 features (MVP default)
    best_at, _, _ = at_sweep(model, val_csv, processed_root, train_csv,
                             device=DEVICE, inference_stride=24, n_features=6)
    print(f'[repro_track] best_at from ft_val = {best_at:.2f}')

    X_tr, y_tr, _      = extract_features_for_split(model, train_csv, processed_root, best_at, 24, DEVICE, 6)
    X_vl, y_vl, _      = extract_features_for_split(model, val_csv,   processed_root, best_at, 24, DEVICE, 6)
    X_te, y_te, te_ids = extract_features_for_split(model, test_csv,  processed_root, best_at, 24, DEVICE, 6)

    sc, pca, lr_m  = fit_lr_model(X_tr, y_tr, C=0.1, use_pca=False)
    val_scores     = predict_lr(X_vl, sc, pca, lr_m)
    test_scores    = predict_lr(X_te, sc, pca, lr_m)
    th_primary, sens, spec = clinical_threshold(y_vl, val_scores)
    print(f'[repro_track] threshold (val): {th_primary:.4f}  sens={sens:.3f}  spec={spec:.3f}')

    repro_df = pd.DataFrame({
        'id': te_ids, 'y_true': y_te, 'y_score': test_scores,
        'best_at': best_at, 'threshold_primary': th_primary,
    })
    repro_df.to_csv(repro_csv, index=False)
    repro_auc = roc_auc_score(y_te, test_scores) if len(np.unique(y_te)) > 1 else 0.
    print(f'[repro_track] AUC on test-55 = {repro_auc:.4f}')
    print(f'[repro_track] Saved -> {repro_csv}')

sys.stdout.flush()

In [ ]:
# ── Cell 16: Export artifacts (Drive / GitHub / local) ────────────────────────
import pathlib, os

ROOT_P       = pathlib.Path(ROOT)
results_base = ROOT_P / 'results'     / 'e2e_cv_v2'
logs_base    = ROOT_P / 'logs'        / 'e2e_cv_v2'
ckpt_base    = ROOT_P / 'checkpoints' / 'e2e_cv_v2'

EXPORT_FILES = [
    results_base / 'global_oof_predictions.csv',
    results_base / 'e2e_cv_v2_final_report.csv',
    results_base / 'e2e_cv_v2_per_fold.csv',
    results_base / 'comparison_table.csv',
    results_base / 'ablation_shared_vs_clean.csv',
    results_base / 'threshold_sweep_summary.csv',
    results_base / 'features_6_vs_12_gate.csv',
    results_base / 'repro_track_55.csv',
    logs_base    / 'split_hashes.txt',
    logs_base    / 'runtime_preflight.md',
    logs_base    / 'incidents.md',
]

# DoD artifact status
print('[export] DoD artifact status:')
for f in EXPORT_FILES:
    status = 'OK     ' if pathlib.Path(f).exists() else 'MISSING'
    print(f'  [{status}] {pathlib.Path(f).name}')

# Google Drive export (Colab only)
if IS_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DRIVE_OUT = '/content/drive/MyDrive/SentinelFatal2/e2e_cv_v2_results'
        os.makedirs(DRIVE_OUT, exist_ok=True)
        for f in EXPORT_FILES:
            fp = pathlib.Path(f)
            if fp.exists():
                sh(f'cp "{fp}" "{DRIVE_OUT}/"')
                print(f'[drive] exported {fp.name}')
        sh(f'zip -qr "{DRIVE_OUT}/e2e_cv_v2_checkpoints.zip" "{ckpt_base}"')
        print(f'[drive] checkpoints bundled -> {DRIVE_OUT}/e2e_cv_v2_checkpoints.zip')
    except Exception as e:
        print(f'[drive] {e} — skipping Drive export')
    try:
        from google.colab import files as colab_files
        for f in EXPORT_FILES:
            if pathlib.Path(f).exists():
                colab_files.download(str(f))
    except Exception:
        pass
else:
    print('[export] Local run — files are on disk.')

# GitHub push (uncomment to enable)
# sh(f'git -C {ROOT} add results/e2e_cv_v2/ logs/e2e_cv_v2/')
# sh(f'git -C {ROOT} commit -m "[auto] E2E CV v2 results"')
# sh(f'git -C {ROOT} push origin master')

print('\n[Cell 16] Export complete.')
sys.stdout.flush()